In [1]:
pip install torch numpy matplotlib

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.10.0+cu128
True
Tesla T4
cuda


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F


In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-05-26 06:25:38--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-05-26 06:25:38 (31.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
#with open("input.txt","r", encoding="utf-8") as f:
  #text=f.read()

#print(text[::300])

text = "I Love You"

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

[' ', 'I', 'L', 'Y', 'e', 'o', 'u', 'v']
8


In [9]:
stoi={ch:i for i, ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])
print(encode("Love"))
print(decode(encode(("Love"))))

[2, 5, 7, 4]
Love


In [10]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:100])

torch.Size([10])
tensor([1, 0, 2, 5, 7, 4, 0, 3, 5, 6])


In [11]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [14]:
batch_size = 2
block_size = 4
n_embd = 64
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  return x.to(device), y.to(device)

x, y = get_batch("train")

print(x)
print(y)

tensor([[5, 7, 4, 0],
        [0, 2, 5, 7]], device='cuda:0')
tensor([[7, 4, 0, 3],
        [2, 5, 7, 4]], device='cuda:0')


In [17]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

In [18]:
class TransformerLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [19]:
model = TransformerLanguageModel().to(device)

In [20]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)
for iter in range(max_iters):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if iter % eval_interval == 0:
        print(loss.item())

2.199404001235962
0.16490143537521362
0.03203282132744789
0.00769246369600296
0.002684536622837186
0.0026288069784641266
0.0007918599294498563
0.001108406693674624
0.0007378214504569769
0.0003951038233935833


In [21]:
context = torch.zeros(
    (1,1),
    dtype=torch.long,
    device=device
)
generated_chars = model.generate(
    context,
    max_new_tokens=500
)[0].tolist()
print(decode(generated_chars))

 Love Yove Yove YoeYYv eYYYYYYYYYYeYY eYYYYYeYYYYYYYYYve Yove Yove Yove Yove YoeYYYYYYYYYYYYv eYYYYYYYYYYve YoeYYYYY YoeYYYYYYv  oove Yove Yove Yove Yove Yove YoeYYYYYYYYeYYYYYYYYYeYYv eYYYYYYYYYYYYYeYYYYYYve YoeYYYYYYYYve Yove Yove YoeYYYYYYYYYYeYYYYYeYYYYYYYYeYY YoeYYYYYYYYYYYYeYYYYeYYYYYYYYYve Yove Yove Yove Yove Yoe YoeYYYYeYY YoeYYYYYYYYYYYYYYYY YoeYYv  oove Yove Yove Yove YoeYYve Yove Yove Yove Yove Yove Yove Yove Yove Yove Yove Yove Yove YoeYYYYYYve Yove YoeYYYYYYYYYv eYYYYYYYYYYv  oove Yo
